# Bounding Box Demo

This notebook loads a CSV dataset with latitude/longitude values, computes a wrapped bounding box that handles antimeridian crossing, exports GeoJSON object, and saves a JPG preview.

## Set Up

In [ ]:
import sys
from pathlib import Path

code_dir = Path("../code").resolve()
sys.path.insert(0, str(code_dir))

print("Using code from:", code_dir)

In [ ]:
# Import Project Code

import time
import json
from IPython.display import Image, display

# Force Python to forget any cached old imports
for module_name in ["bounding_box", "dataset_configs", "set_paths"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

import dataset_configs
import bounding_box as bb

from set_paths import PROJECT_ROOT, OUTPUTS_DIR




In [ ]:
# Verify Imported Code

print("bounding_box module file:")
print(bb.__file__)



## Choose Dataset To Run

In [ ]:
# Datasets that can be run

config_names = [
    name for name in dir(dataset_configs)
    if not name.startswith("__") and isinstance(getattr(dataset_configs, name), dict)
]
print("Names of different datasets that are configured here to be able to run, choose one of these in the next cell: ",config_names)

In [ ]:
# Choose Dataset 

dataset_config = dataset_configs.csv_986627
dataset_config

In [ ]:
# Check Output Folder

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Outputs dir:", OUTPUTS_DIR)

## Run the Bounding Box Code

In [ ]:
# Run Workflow

start_time = time.time()

coords_df = bb.import_dataset_csv(
    dataset_config["csv"],
    dataset_config["skiprows"],
    dataset_config["longitude"],
    dataset_config["latitude"],
    dataset_config["delimiter"]
)

points_gdf = bb.points_for_map(
    coords_df,
    dataset_config["latitude"],
    dataset_config["longitude"]
)

bbox_geojson, ogc_bbox = bb.calc_wrapped_bbox(
    coords_df,
    dataset_config["longitude"],
    dataset_config["latitude"]
)

split_geometries = bb.split_polygon(bbox_geojson)
final_bbox_geojson = bb.build_final_bbox_geojson(split_geometries)

bb.export_geojson(final_bbox_geojson, dataset_config["id"], output_dir=OUTPUTS_DIR)
bb.export_plot_jpg(points_gdf, final_bbox_geojson, dataset_config["id"], output_dir=OUTPUTS_DIR)

bb.log_elapsed_time(start_time)

In [ ]:
# Show JPG

jpg_path = OUTPUTS_DIR / f"output_dataset_{dataset_config['id']}.jpg"
display(Image(filename=str(jpg_path)))

In [ ]:
# Inspect GEOJSON

geojson_path = OUTPUTS_DIR / f"output_dataset_{dataset_config['id']}.geojson"

with open(geojson_path, "r") as f:
    exported_geojson = json.load(f)

exported_geojson